In [ ]:
# !git clone https://github.com/IAMJackYan/Fed-LWR.git
# %cd Fed-LWR


In [ ]:
# !pip install torch torchvision numpy scipy scikit-learn h5py tqdm matplotlib opencv-python SimpleITK medpy pillow

In [1]:
!pip install medpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 3.7 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Created wheel for medpy: filename=MedPy-0.5.2-py3-none-any.whl size=224710 sha256=4f597d9398612e1f2f8c6225792b37b37c6b4a19a7a80594d43617130adbb75f
  Stored in directory: /root/.cache/pip/wheels/d4/33/ed/aaac5a347fb8d41679ca515b8f5c49dfdf49be15bdbb9a905d
Successfully built medpy


In [2]:
import torch
from torchmetrics import Metric


class AccumTensor(Metric):
    def __init__(self, default_value: torch.Tensor):
        super().__init__()

        self.add_state("val", default=default_value, dist_reduce_fx="sum")

    def update(self, input_tensor: torch.Tensor):
        self.val += input_tensor

    def compute(self):
        return self.val

In [3]:
"""
Helper for CKA in PyTorch.
Adds hooks to modules of a given model.

Repo: https://github.com/numpee/CKA.pytorch
Author: Dongwan Kim (Github: Numpee)
Year: 2022
"""

from typing import Optional, Union, Callable, Tuple, Type, List

import torch
from torch import nn as nn
from torchvision.models.resnet import Bottleneck, BasicBlock

# _HOOK_LAYER_TYPES = (
#     Bottleneck, BasicBlock, nn.Conv2d, nn.AdaptiveAvgPool2d, nn.MaxPool2d, nn.modules.batchnorm._BatchNorm, nn.ConvTranspose2d, nn.BatchNorm2d)

_HOOK_LAYER_TYPES = (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)

class HookManager:
    def __init__(self, model: nn.Module, hook_fn: Optional[Union[str, Callable]] = None,
                 hook_layer_types: Tuple[Type[nn.Module], ...] = _HOOK_LAYER_TYPES,
                 calculate_gram: bool = True) -> None:
        """
        Add hooks to models.
        Mainly supports ResNets.
        :param model: model to attach hooks to
        :param hook_fn: the hook function or string. Options: ("avgpool", "flatten"). Default: flatten
        :param hook_layer_types: layer types to register hooks. Should be nn.Module
        """
        self.model = model
        self.hook_fn = hook_fn
        self.hook_layer_types = hook_layer_types
        self.calculate_gram = calculate_gram
        for layer in self.hook_layer_types:
            if not issubclass(layer, nn.Module):
                raise TypeError(f"Class {layer} is not an nn.Module.")

        if self.hook_fn is None:
            self.hook_fn = self.flatten_hook_fn
            print("No hook function provided. Using flatten_hook_fn.")
        elif type(self.hook_fn) == str:
            hook_fn_dict = {'flatten': self.flatten_hook_fn, 'avgpool': self.avgpool_hook_fn}
            if self.hook_fn in hook_fn_dict:
                self.hook_fn = hook_fn_dict[self.hook_fn]
            else:
                raise ValueError(f"No hook function named {self.hook_fn}. Options: {list(hook_fn_dict.keys())}")

        # Not using dictionary because a single module may be used multiple times in forward
        self.features = []
        self.module_names = []
        self.handles = []

        self.register_hooks(self.hook_fn)

    def get_features(self) -> List[torch.Tensor]:
        return self.features

    def get_module_names(self) -> List[str]:
        return self.module_names

    def clear_features(self) -> None:
        self.features = []
        self.module_names = []

    def clear_all(self) -> None:
        self.clear_hooks()
        self.clear_features()

    def clear_hooks(self) -> None:
        num_handles = len(self.handles)
        for handle in self.handles:
            handle.remove()
        self.handles = []
        for m in self.model.modules():
            if hasattr(m, 'module_name'):
                delattr(m, 'module_name')
        print(f"{num_handles} handles removed.")

    def register_hooks(self, hook_fn: Callable) -> None:
        prev_num_handles = len(self.handles)
        self._register_hook_recursive(self.model, hook_fn, prev_name="")
        new_num_handles = len(self.handles)
        print(f"{new_num_handles - prev_num_handles} Hooks registered. Total hooks: {new_num_handles}")

    def _register_hook_recursive(self, module: nn.Module, hook_fn: Callable, prev_name: str = "") -> None:
        for name, child in module.named_children():
            curr_name = f"{prev_name}.{name}" if prev_name else name
            curr_name = curr_name.replace("_model.", "")
            num_grandchildren = len(list(child.children()))
            if num_grandchildren > 0:
                self._register_hook_recursive(child, hook_fn, prev_name=curr_name)
            if isinstance(child, self.hook_layer_types):
                handle = child.register_forward_hook(hook_fn)
                self.handles.append(handle)
                setattr(child, 'module_name', curr_name)

    def flatten_hook_fn(self, module: nn.Module, inp: torch.Tensor, out: torch.Tensor) -> None:
        batch_size = out.size(0)
        feature = out.reshape(batch_size, -1)
        if self.calculate_gram:
            feature = gram(feature)
        module_name = getattr(module, 'module_name')
        self.features.append(feature)
        self.module_names.append(module_name)

    def avgpool_hook_fn(self, module: nn.Module, inp: torch.Tensor, out: torch.Tensor) -> None:
        if out.dim() == 4:
            feature = out.mean(dim=(-1, -2))
        elif out.dim() == 3:
            feature = out.mean(dim=-1)
        else:
            feature = out
        if self.calculate_gram:
            feature = gram(feature)
        module_name = getattr(module, 'module_name')
        self.features.append(feature)
        self.module_names.append(module_name)


def gram(x: torch.Tensor) -> torch.Tensor:
    return x.matmul(x.t())

In [4]:

import time
import os
import logging
import torch
import numpy as np

from pathlib import Path
from sklearn.metrics import confusion_matrix
from torch import nn
from medpy import metric

def set_for_logger(args):
    log_filepath = os.path.join(args.log_dir, args.experiment + '.txt')

    Path(args.log_dir).mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    formatter = logging.Formatter(
        '%(asctime)s ===> %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S')

    # args FileHandler to save log file
    fh = logging.FileHandler(log_filepath)
    fh.setLevel(logging.INFO)
    fh.setFormatter(formatter)

    # args StreamHandler to print log to console
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(formatter)

    # add the two Handler
    logger.addHandler(ch)
    logger.addHandler(fh)


def calculate_metric_perimage(pred, gt):
    pred[pred > 0] = 1
    gt[gt > 0] = 1
    if pred.sum() > 0 and gt.sum()>0:
        dice = metric.binary.dc(pred, gt)
        hd95 = metric.binary.hd95(pred, gt)
        return dice, hd95
    elif pred.sum() > 0 and gt.sum()==0:
        return 1, 0
    else:
        return 0, 0


def dice_func(output, label):
    softmax_pred = torch.nn.functional.softmax(output, dim=1)
    seg_pred = torch.argmax(softmax_pred, dim=1)
    all_dice = 0
    label = label.squeeze(dim=1)
    batch_size = label.shape[0]
    num_class = softmax_pred.shape[1]
    for i in range(num_class):
        each_pred = torch.zeros_like(seg_pred)
        each_pred[seg_pred == i] = 1

        each_gt = torch.zeros_like(label)
        each_gt[label == i] = 1

        intersection = torch.sum((each_pred * each_gt).view(batch_size, -1), dim=1)

        union = each_pred.view(batch_size, -1).sum(1) + each_gt.view(batch_size, -1).sum(1)
        dice = (2. * intersection) / (union + 1e-5)

        all_dice += torch.mean(dice)

    return all_dice.item() * 1.0 / num_class


In [5]:
"""
Tool to compute Centered Kernel Alignment (CKA) in PyTorch w/ GPU (single or multi).

Repo: https://github.com/numpee/CKA.pytorch
Author: Dongwan Kim (Github: Numpee)
Year: 2022
"""

from __future__ import annotations

from typing import Tuple, Optional, Callable, Type, Union, TYPE_CHECKING, List

import torch
import torch.nn as nn
from tqdm.autonotebook import tqdm

# from hook_manager import HookManager, _HOOK_LAYER_TYPES
# from metrics import AccumTensor

if TYPE_CHECKING:
    from torch.utils.data import DataLoader


class CKACalculator:
    def __init__(self, model1: nn.Module, model2: nn.Module, dataloader: DataLoader,
                 hook_fn: Optional[Union[str, Callable]] = None,
                 hook_layer_types: Tuple[Type[nn.Module], ...] = _HOOK_LAYER_TYPES, num_epochs: int = 10,
                 group_size: int = 512, epsilon: float = 1e-4, is_main_process: bool = True) -> None:
        """
        Class to extract intermediate features and calculate CKA Matrix.
        :param model1: model to evaluate. __call__ function should be implemented if NOT instance of `nn.Module`.
        :param model2: second model to evaluate. __call__ function should be implemented if NOT instance of `nn.Module`.
        :param dataloader: Torch DataLoader for dataloading. Assumes first return value contains input images.
        :param hook_fn: Optional - Hook function or hook name string for the HookManager. Options: [flatten, avgpool]. Default: flatten
        :param hook_layer_types: Types of layers (modules) to add hooks to.
        :param num_epochs: Number of epochs for cka_batch. Default: 10
        :param group_size: group_size for GPU acceleration. Default: 512
        :param epsilon: Small multiplicative value for HSIC. Default: 1e-4
        :param is_main_process: is current instance main process. Default: True
        """
        self.model1 = model1
        self.model2 = model2
        self.dataloader = dataloader
        self.num_epochs = num_epochs
        self.group_size = group_size
        self.epsilon = epsilon
        self.is_main_process = is_main_process

        self.model1.eval()
        self.model2.eval()
        self.hook_manager1 = HookManager(self.model1, hook_fn, hook_layer_types, calculate_gram=True)
        self.hook_manager2 = HookManager(self.model2, hook_fn, hook_layer_types, calculate_gram=True)
        self.module_names_X = None
        self.module_names_Y = None
        self.num_layers_X = None
        self.num_layers_Y = None
        self.num_elements = None

        # Metrics to track
        self.cka_matrix = None
        self.hsic_matrix = None
        self.self_hsic_x = None
        self.self_hsic_y = None

    @torch.no_grad()
    def calculate_cka_matrix(self) -> torch.Tensor:
        curr_hsic_matrix = None
        curr_self_hsic_x = None
        curr_self_hsic_y = None
        for epoch in range(self.num_epochs):
            loader = tqdm(self.dataloader, desc=f"Epoch {epoch}", disable=not self.is_main_process)
            for it, (imgs, *_) in enumerate(loader):
                imgs = imgs.cuda(non_blocking=True)
                self.model1(imgs)
                self.model2(imgs)
                all_layer_X, all_layer_Y = self.extract_layer_list_from_hook_manager()

                # Initialize values on first loop
                if self.num_layers_X is None:
                    curr_hsic_matrix, curr_self_hsic_x, curr_self_hsic_y = self._init_values(all_layer_X, all_layer_Y)

                # Get self HSIC values --> HSIC(K, K), HSIC(L, L)
                self._calculate_self_hsic(all_layer_X, all_layer_Y, curr_self_hsic_x, curr_self_hsic_y)

                # Get cross HSIC values --> HSIC(K, L)
                self._calculate_cross_hsic(all_layer_X, all_layer_Y, curr_hsic_matrix)

                self.hook_manager1.clear_features()
                self.hook_manager2.clear_features()
                curr_hsic_matrix.fill_(0)
                curr_self_hsic_x.fill_(0)
                curr_self_hsic_y.fill_(0)

        # Update values across GPUs
        hsic_matrix = self.hsic_matrix.compute()
        hsic_x = self.self_hsic_x.compute()
        hsic_y = self.self_hsic_y.compute()

        if torch.isnan(hsic_x).any():
            print('hsic_x')
        
        if torch.isnan(hsic_y).any():
            print('hsic_y')

        if torch.isnan(hsic_x * hsic_y).any():
            print('hsic_x * hsic_y ' )


        hsic_multi = hsic_x * hsic_y
        hsic_multi = hsic_multi.clip(min=self.epsilon)

        self.cka_matrix = hsic_matrix.reshape(self.num_layers_Y, self.num_layers_X) / (torch.sqrt(hsic_multi + self.epsilon))
        
        if torch.isnan(torch.sqrt(hsic_multi + self.epsilon)).any():
            print('torch.sqrt(hsic_multi)')

        # print(self.cka_matrix.diagonal())
        # self.cka_matrix = self.cka_matrix.flip(0)
        return self.cka_matrix

    def extract_layer_list_from_hook_manager(self) -> Tuple[List, List]:
        all_layer_X, all_layer_Y = self.hook_manager1.get_features(), self.hook_manager2.get_features()
        return all_layer_X, all_layer_Y

    def hsic1(self, K: torch.Tensor, L: torch.Tensor) -> torch.Tensor:
        '''
        Batched version of HSIC.
        :param K: Size = (B, N, N) where N is the number of examples and B is the group/batch size
        :param L: Size = (B, N, N) where N is the number of examples and B is the group/batch size
        :return: HSIC tensor, Size = (B)
        '''
        assert K.size() == L.size()
        assert K.dim() == 3
        K = K.clone()
        L = L.clone()
        n = K.size(1)

        # K, L --> K~, L~ by setting diagonals to zero
        K.diagonal(dim1=-1, dim2=-2).fill_(0)
        L.diagonal(dim1=-1, dim2=-2).fill_(0)

        KL = torch.bmm(K, L)
        trace_KL = KL.diagonal(dim1=-1, dim2=-2).sum(-1).unsqueeze(-1).unsqueeze(-1)
        middle_term = K.sum((-1, -2), keepdim=True) * L.sum((-1, -2), keepdim=True)
        middle_term /= (n - 1) * (n - 2)
        right_term = KL.sum((-1, -2), keepdim=True)
        right_term *= 2 / (n - 2)
        main_term = trace_KL + middle_term - right_term

        if torch.isnan(trace_KL).any():
            print('trace_KL')
        
        if torch.isnan(middle_term).any():
            print('middle_term')
        
        if torch.isnan(right_term).any():
            print('right_term')
        
        if torch.isnan(main_term).any():
            print('main_term')

        hsic = main_term / (n ** 2 - 3 * n)

        return hsic.squeeze(-1).squeeze(-1)

    def reset(self) -> None:
        # Set values to none, clear feature and hooks
        self.cka_matrix = None
        self.hsic_matrix = None
        self.self_hsic_x = None
        self.self_hsic_y = None
        self.hook_manager1.clear_all()
        self.hook_manager2.clear_all()

    def _init_values(self, all_layer_X, all_layer_Y):
        self.num_layers_X = len(all_layer_X)
        self.num_layers_Y = len(all_layer_Y)
        self.module_names_X = self.hook_manager1.get_module_names()
        self.module_names_Y = self.hook_manager2.get_module_names()
        self.num_elements = self.num_layers_Y * self.num_layers_X
        curr_hsic_matrix = torch.zeros(self.num_elements).cuda()
        curr_self_hsic_x = torch.zeros(1, self.num_layers_X).cuda()
        curr_self_hsic_y = torch.zeros(self.num_layers_Y, 1).cuda()
        self.hsic_matrix = AccumTensor(torch.zeros_like(curr_hsic_matrix)).cuda()
        self.self_hsic_x = AccumTensor(torch.zeros_like(curr_self_hsic_x)).cuda()
        self.self_hsic_y = AccumTensor(torch.zeros_like(curr_self_hsic_y)).cuda()
        return curr_hsic_matrix, curr_self_hsic_x, curr_self_hsic_y

    def _calculate_self_hsic(self, all_layer_X, all_layer_Y, curr_self_hsic_x, curr_self_hsic_y):
        for start_idx in range(0, self.num_layers_X, self.group_size):
            end_idx = min(start_idx + self.group_size, self.num_layers_X)
            K = torch.stack([all_layer_X[i] for i in range(start_idx, end_idx)], dim=0)
            curr_self_hsic_x[0, start_idx:end_idx] += self.hsic1(K, K) * self.epsilon
        for start_idx in range(0, self.num_layers_Y, self.group_size):
            end_idx = min(start_idx + self.group_size, self.num_layers_Y)
            L = torch.stack([all_layer_Y[i] for i in range(start_idx, end_idx)], dim=0)
            curr_self_hsic_y[start_idx:end_idx, 0] += self.hsic1(L, L) * self.epsilon
        

        # if torch.isnan(curr_self_hsic_x).any():
        #     print('curr_self_hsic_x')
        
        # if torch.isnan(curr_self_hsic_y).any():
        #     print('curr_self_hsic_y')

        self.self_hsic_x.update(curr_self_hsic_x)
        self.self_hsic_y.update(curr_self_hsic_y)

    def _calculate_cross_hsic(self, all_layer_X, all_layer_Y, curr_hsic_matrix):
        for start_idx in range(0, self.num_elements, self.group_size):
            end_idx = min(start_idx + self.group_size, self.num_elements)
            K = torch.stack([all_layer_X[i % self.num_layers_X] for i in range(start_idx, end_idx)], dim=0)
            L = torch.stack([all_layer_Y[j // self.num_layers_X] for j in range(start_idx, end_idx)], dim=0)
            curr_hsic_matrix[start_idx:end_idx] += self.hsic1(K, L) * self.epsilon
        self.hsic_matrix.update(curr_hsic_matrix)


def gram(x: torch.Tensor) -> torch.Tensor:
    return x.matmul(x.t())

In [6]:
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0, activation='sigmoid'):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
        self.activation = activation

    def dice_coef(self, pred, gt):
        softmax_pred = torch.nn.functional.softmax(pred, dim=1)
        seg_pred = torch.argmax(softmax_pred, dim=1)
        all_dice = 0
        gt = gt.squeeze(dim=1)
        batch_size = gt.shape[0]
        num_class = softmax_pred.shape[1]
        for i in range(num_class):

            each_pred = torch.zeros_like(seg_pred)
            each_pred[seg_pred==i] = 1

            each_gt = torch.zeros_like(gt)
            each_gt[gt==i] = 1            

        
            intersection = torch.sum((each_pred * each_gt).view(batch_size, -1), dim=1)
            
            union = each_pred.view(batch_size,-1).sum(1) + each_gt.view(batch_size,-1).sum(1)
            dice = (2. *  intersection )/ (union + 1e-5)
         
            all_dice += torch.mean(dice)
 
        return all_dice * 1.0 / num_class


    def forward(self, pred, gt):
        sigmoid_pred = F.softmax(pred,dim=1)

        batch_size = gt.shape[0]
        num_class = sigmoid_pred.shape[1]
    
        bg = torch.zeros_like(gt)
        bg[gt==0] = 1
        label1 = torch.zeros_like(gt)
        label1[gt==1] = 1
        label2 = torch.zeros_like(gt)
        label2[gt == 2] = 1
        label = torch.cat([bg, label1, label2], dim=1)
        
        loss = 0
        smooth = 1e-5

        for i in range(num_class):
            intersect = torch.sum(sigmoid_pred[:, i, ...] * label[:, i, ...])
            z_sum = torch.sum(sigmoid_pred[:, i, ...] )
            y_sum = torch.sum(label[:, i, ...] )
            loss += (2 * intersect + smooth) / (z_sum + y_sum + smooth)
        loss = 1 - loss * 1.0 / num_class
        return loss

class JointLoss(nn.Module):
    def __init__(self):
        super(JointLoss, self).__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss()

    def forward(self, pred, gt):
        ce =  self.ce(pred, gt.squeeze(axis=1).long())
        return (ce + self.dice(pred, gt))/2

In [7]:
import os
import torch
import numpy as np
from glob import glob
from torch.utils.data import Dataset
import h5py
import itertools
import torch.nn.functional as F
from torch.utils.data.sampler import Sampler
import random
from scipy import ndimage
from scipy.ndimage import _ni_support
from scipy.ndimage.morphology import distance_transform_edt, binary_erosion,\
    generate_binary_structure


class RIF(Dataset):
    """ LA Dataset """
    def __init__(self, client_idx=None, base_path=None, split='train', transform=None, isVal=0):
        self.root_dir = base_path  # todo
        self.transform = transform
        self.split = split
        self.client_name = ['Domain1', 'Domain2', 'Domain3', 'Domain4']
        self.image_list = glob(
            self.root_dir +
            '/{}/{}/image/*'.format(self.client_name[client_idx], split))

        if split == 'train' and isVal==0:
            train_split = int(len(self.image_list) * 0.8)
            self.image_list = self.image_list[:train_split]
        elif split == 'train' and isVal==1:
            train_split = int(len(self.image_list) * 0.8)
            self.image_list = self.image_list[train_split:]

        print("total {} slices".format(len(self.image_list)))



    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        idx = random.randint(0, len(self.image_list) - 1)
        raw_file = self.image_list[idx]
        image = np.load(raw_file).transpose((2, 0, 1))
        mask = np.load(raw_file.replace('image', 'mask'))
        image = torch.from_numpy(image) / 255.


        # _mask = []
        # for _c in range(2):
        #     _mask.append((mask > _c).copy())
        mask = torch.from_numpy(np.array(mask)).unsqueeze(0)
        sample = {"image": image, "label": mask}
        if self.transform is not None:
            sample = self.transform(sample)
        return sample["image"], sample["label"]


class CenterCrop(object):
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, label = sample['image'], sample['label']

        # pad the sample if necessary
        if label.shape[0] <= self.output_size[0] or label.shape[1] <= self.output_size[1] or label.shape[2] <= \
                self.output_size[2]:
            pw = max((self.output_size[0] - label.shape[0]) // 2 + 3, 0)
            ph = max((self.output_size[1] - label.shape[1]) // 2 + 3, 0)
            pd = max((self.output_size[2] - label.shape[2]) // 2 + 3, 0)
            image = np.pad(image, [(pw, pw), (ph, ph), (pd, pd)],
                           mode='constant',
                           constant_values=0)
            label = np.pad(label, [(pw, pw), (ph, ph), (pd, pd)],
                           mode='constant',
                           constant_values=0)

        (w, h, d) = image.shape

        w1 = int(round((w - self.output_size[0]) / 2.))
        h1 = int(round((h - self.output_size[1]) / 2.))
        d1 = int(round((d - self.output_size[2]) / 2.))

        label = label[w1:w1 + self.output_size[0], h1:h1 + self.output_size[1],
                      d1:d1 + self.output_size[2]]
        image = image[w1:w1 + self.output_size[0], h1:h1 + self.output_size[1],
                      d1:d1 + self.output_size[2]]

        return {'image': image, 'label': label}


class RandomNoise(object):
    def __init__(self, mu=0, sigma=0.4):
        self.mu = mu
        self.sigma = sigma

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        noise = torch.clamp(
            torch.rand(image.size()) * self.sigma, -2 * self.sigma,
            2 * self.sigma)
        noise = noise + self.mu
        image = image + noise
        return {'image': image, 'label': label}


# if __name__ == '__main__':
#     for _id in range(6):
#         x = Dataset(_id, 'train')
#         y = Dataset(_id, 'test')
#         n = len(x.image_list) + len(y.image_list)
#         print(_id, n)

/tmp/ipykernel_48/383704839.py:13: DeprecationWarning: Please import `distance_transform_edt` from the `scipy.ndimage` namespace; the `scipy.ndimage.morphology` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.morphology import distance_transform_edt, binary_erosion,\
/tmp/ipykernel_48/383704839.py:13: DeprecationWarning: Please import `binary_erosion` from the `scipy.ndimage` namespace; the `scipy.ndimage.morphology` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.morphology import distance_transform_edt, binary_erosion,\
/tmp/ipykernel_48/383704839.py:13: DeprecationWarning: Please import `generate_binary_structure` from the `scipy.ndimage` namespace; the `scipy.ndimage.morphology` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.morphology import distance_transform_edt, binary_erosion,\


In [19]:
# from .rif import RIF
import os
from torch.utils.data import DataLoader
import logging
import torch



def build_dataloader(args, clients):

    train_dls = []
    val_dls = []
    test_dls = []

    dataset_lens = []

    for idx, client in enumerate(clients):
        train_set = RIF(client_idx=idx, base_path=args.data_root,
                             split='train', transform=None, isVal=0)
        valid_set = RIF(client_idx=idx, base_path=args.data_root,
                             split='train', transform=None, isVal=1)
        test_set = RIF(client_idx=idx, base_path=args.data_root,
                             split='test', transform=None)
 
        logging.info('{} train  dataset: {}'.format(client, len(train_set)))
        logging.info('{} val  dataset: {}'.format(client, len(valid_set)))
        logging.info('{} test  dataset: {}'.format(client, len(test_set)))
 

        train_loader = torch.utils.data.DataLoader(train_set, batch_size=args.batch_size,
                                               shuffle=True, drop_last=True)
        valid_loader = torch.utils.data.DataLoader(valid_set, batch_size=args.batch_size,
                                               shuffle=False, drop_last=True)
        test_loader = torch.utils.data.DataLoader(test_set, batch_size=args.batch_size,
                                              shuffle=False, drop_last=True)

        train_dls.append(train_loader)
        val_dls.append(valid_loader)
        test_dls.append(test_loader)

        dataset_lens.append(len(train_set))
    
    client_weight = []
    total_len = sum(dataset_lens)
    for i in dataset_lens:
        client_weight.append(i / total_len)

    return train_dls, val_dls, test_dls, client_weight


In [9]:

from collections import OrderedDict

import torch
import torch.nn as nn
import numpy as np

import torch.nn.functional as F

def _block(in_channels, features, name, affine=True, track_running_stats=True):
    bn_func = nn.BatchNorm2d

    return nn.Sequential(
        OrderedDict(
            [
                (
                    name + "_conv1",
                    nn.Conv2d(
                        in_channels=in_channels,
                        out_channels=features,
                        kernel_size=3,
                        padding=1,
                        bias=False,
                    ),
                ),
                (name + "_bn1", bn_func(num_features=features, affine=affine,
                                        track_running_stats=track_running_stats)),
                (name + "_relu1", nn.ReLU(inplace=True)),
                (
                    name + "_conv2",
                    nn.Conv2d(
                        in_channels=features,
                        out_channels=features,
                        kernel_size=3,
                        padding=1,
                        bias=False,
                    ),
                ),
                (name + "_bn2", bn_func(num_features=features, affine=affine,
                                        track_running_stats=track_running_stats)),
                (name + "_relu2", nn.ReLU(inplace=True)),
            ]
        )
    )



class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2, init_features=32,
                 affine=True, track_running_stats=True, aug_method=None):
        super(UNet, self).__init__()

        features = init_features
        self.encoder1 = _block(in_channels, features, name="enc1", affine=affine,
                               track_running_stats=track_running_stats)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = _block(features, features * 2, name="enc2", affine=affine,
                               track_running_stats=track_running_stats)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder3 = _block(features * 2, features * 4, name="enc3", affine=affine,
                               track_running_stats=track_running_stats)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder4 = _block(features * 4, features * 8, name="enc4", affine=affine,
                               track_running_stats=track_running_stats)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = _block(features * 8, features * 16, name="bottleneck", affine=affine,
                                 track_running_stats=track_running_stats)

        self.upconv4 = nn.ConvTranspose2d(
            features * 16, features * 8, kernel_size=2, stride=2
        )
        self.decoder4 = _block((features * 8) * 2, features * 8, name="dec4", affine=affine,
                               track_running_stats=track_running_stats)

        self.upconv3 = nn.ConvTranspose2d(
            features * 8, features * 4, kernel_size=2, stride=2,
        )
        self.decoder3 = _block((features * 4) * 2, features * 4, name="dec3", affine=affine,
                               track_running_stats=track_running_stats)
        self.upconv2 = nn.ConvTranspose2d(
            features * 4, features * 2, kernel_size=2, stride=2
        )
        self.decoder2 = _block((features * 2) * 2, features * 2, name="dec2", affine=affine,
                               track_running_stats=track_running_stats)

        self.upconv1 = nn.ConvTranspose2d(
            features * 2, features, kernel_size=2, stride=2,
        )
        self.decoder1 = _block(features * 2, features, name="dec1", affine=affine,
                               track_running_stats=track_running_stats)

        self.conv = nn.Conv2d(
            in_channels=features, out_channels=out_channels, kernel_size=1
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc1_ = self.pool1(enc1)

        enc2 = self.encoder2(enc1_)
        enc2_ = self.pool2(enc2)

        enc3 = self.encoder3(enc2_)
        enc3_ = self.pool3(enc3)

        enc4 = self.encoder4(enc3_)
        enc4_ = self.pool4(enc4)

        bottleneck = self.bottleneck(enc4_)

        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)

        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        dec1 = self.conv(dec1)

        return dec1



class UNet_pro(nn.Module):
    def __init__(self, in_channels=3, out_channels=2, init_features=32,
                 affine=True, track_running_stats=True, aug_method=None):
        super(UNet_pro, self).__init__()

        features = init_features
        self.encoder1 = _block(in_channels, features, name="enc1", affine=affine,
                               track_running_stats=track_running_stats)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = _block(features, features * 2, name="enc2", affine=affine,
                               track_running_stats=track_running_stats)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder3 = _block(features * 2, features * 4, name="enc3", affine=affine,
                               track_running_stats=track_running_stats)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder4 = _block(features * 4, features * 8, name="enc4", affine=affine,
                               track_running_stats=track_running_stats)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = _block(features * 8, features * 16, name="bottleneck", affine=affine,
                                 track_running_stats=track_running_stats)

        self.upconv4 = nn.ConvTranspose2d(
            features * 16, features * 8, kernel_size=2, stride=2
        )
        self.decoder4 = _block((features * 8) * 2, features * 8, name="dec4", affine=affine,
                               track_running_stats=track_running_stats)

        self.upconv3 = nn.ConvTranspose2d(
            features * 8, features * 4, kernel_size=2, stride=2,
        )
        self.decoder3 = _block((features * 4) * 2, features * 4, name="dec3", affine=affine,
                               track_running_stats=track_running_stats)
        self.upconv2 = nn.ConvTranspose2d(
            features * 4, features * 2, kernel_size=2, stride=2
        )
        self.decoder2 = _block((features * 2) * 2, features * 2, name="dec2", affine=affine,
                               track_running_stats=track_running_stats)

        self.upconv1 = nn.ConvTranspose2d(
            features * 2, features, kernel_size=2, stride=2,
        )
        self.decoder1 = _block(features * 2, features, name="dec1", affine=affine,
                               track_running_stats=track_running_stats)

        self.conv = nn.Conv2d(
            in_channels=features, out_channels=out_channels, kernel_size=1
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc1_ = self.pool1(enc1)

        enc2 = self.encoder2(enc1_)
        enc2_ = self.pool2(enc2)

        enc3 = self.encoder3(enc2_)
        enc3_ = self.pool3(enc3)

        enc4 = self.encoder4(enc3_)
        enc4_ = self.pool4(enc4)
        
        
        bottleneck = self.bottleneck(enc4_)

        z = F.adaptive_avg_pool2d(bottleneck,2).view(bottleneck.shape[0],-1)
        #z = bottleneck 

        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)

        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        shadow = dec1
        dec1 = self.conv(dec1)

        return dec1, z, shadow

In [10]:
import logging
import copy
# from .unet import UNet,UNet_pro

def build_model(args, clients, device):

    n_classes = 2

    if args.dataset=='pmri':
        n_classes = 3 

    if args.model == 'unet':
        model = UNet(out_channels=n_classes)
    elif args.model == 'unet_pro':
        model = UNet_pro(out_channels=n_classes)
    else:
        logging.info('unknow model')
        return

    model = model.to(device)

    local_models = []
    for id, c in enumerate(clients):
        

        local_models.append(copy.deepcopy(model))

        total_params = sum(p.numel() for p in model.parameters())
        logging.info('{}: model parameters {} M'.format(c, total_params/1024/1024))

    global_model = copy.deepcopy(model)
    return local_models, global_model


In [20]:
import logging
import torch
import os
import numpy as np
import random
import argparse

from pathlib import Path
import torch.nn.functional as F

import copy

@torch.no_grad()
def cal_contribution_of_clients(global_model, local_models, train_loaders, device, layer_names):

    for model in local_models:
        model.eval()

    global_model.eval()

    lay_wise_weight = {}

    for cid, loader in enumerate(train_loaders):
        calculator = CKACalculator(model1=global_model, model2=local_models[cid], dataloader=loader, hook_fn='flatten', num_epochs=5)
        cka_output = calculator.calculate_cka_matrix()

        lay_index = cka_output.shape[0]

        for i in range(lay_index):
            if i not in lay_wise_weight.keys():
                lay_wise_weight[i] = [cka_output[i][i].cpu().item()]
            else:
                lay_wise_weight[i].append(cka_output[i][i].cpu().item())
        
        calculator.reset()
        torch.cuda.empty_cache()
    
    lay_wise_weight_map = {}
    for i in range(lay_index):
        lay_wise_weight[i] = np.clip(lay_wise_weight[i], a_min=1e-4, a_max=0.99)
        lay_wise_weight[i] = 1 - lay_wise_weight[i]
        lay_wise_weight[i] /=  (np.sum(lay_wise_weight[i]) + 1e-9)
        lay_wise_weight_map[layer_names[i]] = lay_wise_weight[i]

    return lay_wise_weight_map      


def get_args():
    parser = argparse.ArgumentParser()

    parser.add_argument('--seed', type=int, default=0, help="Random seed")
    parser.add_argument('--data_root', type=str, required=False,
                    default='/kaggle/input/fundus/fundus',
                    help="Data directory")

    parser.add_argument('--dataset', type=str, default='fundus')
    parser.add_argument('--model', type=str, default='unet')

    parser.add_argument('--rounds', type=int, default=200, help='number of maximum communication round')
    parser.add_argument('--epochs', type=int, default=1, help='number of local epochs')
    parser.add_argument('--device', type=str, default='cuda', help='The device to run the program')

    parser.add_argument('--log_dir', type=str, required=False, default="./logs/", help='Log directory path')
    parser.add_argument('--save_dir', type=str, required=False, default="./weights/", help='Log directory path')

    parser.add_argument('--lr', type=float, default=1e-3, help='learning rate (default: 0.1)')
    parser.add_argument('--weight_decay', type=float, default=1e-4, help="L2 regularization strength")
    parser.add_argument('--batch-size', type=int, default=8, help='input batch size for training (default: 64)')
    parser.add_argument('--experiment', type=str, default='experiment', help='The device to run the program')

    parser.add_argument('--test_step', type=int, default=1)
    parser.add_argument('--train_ratio', type=float, default=0.6, help="")
    parser.add_argument('--cka_type', type=str, default='linear')

    args = parser.parse_args([])
    return args

def init_model_weight(model, weight):
    for key in weight.keys():
        if key not in model.state_dict().keys():
            pass
        else:
            model.state_dict()[key].data.copy_(weight[key])


def communication(server_model, models, client_weights):
    with torch.no_grad():
        for key in server_model.state_dict().keys():
            temp = torch.zeros_like(server_model.state_dict()[key], dtype=torch.float32)
            for client_idx in range(len(client_weights)):
                temp += client_weights[client_idx] * models[client_idx].state_dict()[key]
            server_model.state_dict()[key].data.copy_(temp)
            for client_idx in range(len(client_weights)):
                models[client_idx].state_dict()[key].data.copy_(server_model.state_dict()[key])
    return server_model, models



def communication_ours(server_model, models, lay_wise_client_weights):
    with torch.no_grad():
        for lay_id, key in enumerate(server_model.state_dict().keys()):

            t = key.split('.')
            if len(t) > 2:
                l_name = t[0] + '.' + t[1]
            else:
                l_name = t[0]
            
            client_weights = lay_wise_client_weights[l_name]
            temp = torch.zeros_like(server_model.state_dict()[key], dtype=torch.float32)

            for client_idx in range(len(client_weights)):
                temp += client_weights[client_idx] * models[client_idx].state_dict()[key]

            server_model.state_dict()[key].data.copy_(temp)
            for client_idx in range(len(client_weights)):
                models[client_idx].state_dict()[key].data.copy_(server_model.state_dict()[key])
    return server_model, models


def train(cid, model, dataloader, device, optimizer, epochs, loss_func):
    model.train()

    for epoch in range(epochs):
        train_acc = 0.
        loss_all = 0.
        for x, target in dataloader:

            x = x.to(device)

            target = target.to(device)

            output = model(x)
            
            optimizer.zero_grad()

            loss = loss_func(output, target)
            loss_all += loss.item()

            train_acc += DiceLoss().dice_coef(output, target).item()

            loss.backward()
            optimizer.step()

        avg_loss = loss_all / len(dataloader)
        train_acc = train_acc / len(dataloader)
        logging.info('Client: [%d]  Epoch: [%d]  train_loss: %f train_acc: %f'%(cid, epoch, avg_loss, train_acc))

def get_model_layer_name(model):
    layer_names = []
    for name in model.state_dict().keys():
        t = name.split('.')
        if len(t) > 2:
            l_name = t[0] + '.' + t[1]
        else:
            l_name = t[0]
        if l_name not in layer_names:
            layer_names.append(l_name)
    return layer_names


def test(model, dataloader, device, loss_func):
    model.eval()

    loss_all = 0
    test_acc = 0

    with torch.no_grad():
        for x, target in dataloader:

            x = x.to(device)
            target = target.to(device)

            output = model(x)
            loss = loss_func(output, target)
            loss_all += loss.item()

            test_acc += DiceLoss().dice_coef(output, target).item()
        

    acc = test_acc/ len(dataloader)
    loss = loss_all / len(dataloader)

    return loss, acc


def main(args):
    set_for_logger(args)
    logging.info(args)

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(args.seed)

    device = torch.device(args.device)

    clients = ['Domain1', 'Domain2', 'Domain3', 'Domain4']

    # build dataset
    train_dls, val_dls, test_dls, client_weight = build_dataloader(args, clients)

    # build model
    local_models, global_model = build_model(args, clients, device)

    layer_names = get_model_layer_name(global_model)

    # build loss
    loss_fun = DiceLoss()

    optimizer = []
    for id in range(len(clients)):
        optimizer.append(torch.optim.Adam(local_models[id].parameters(), lr=args.lr, weight_decay=args.weight_decay, betas=(0.9, 0.99)))

    best_dice = 0
    best_dice_round = 0
    best_local_dice = []

    weight_save_dir = os.path.join(args.save_dir, args.experiment)
    Path(weight_save_dir).mkdir(parents=True, exist_ok=True)
    logging.info('checkpoint will be saved at {}'.format(weight_save_dir))

    client_weight = []
    for idx, client in enumerate(clients):
        client_weight.append(1/len(clients))
    
    
    for r in range(args.rounds):

        logging.info('-------- Commnication Round: %3d --------'%r)

        global_w = global_model.state_dict()

        for idx, client in enumerate(clients):
            train(idx, local_models[idx], train_dls[idx], device, optimizer[idx], args.epochs, loss_fun)
            

        temp_locals = copy.deepcopy(local_models)

        #commnication
        communication(global_model, local_models, client_weight)

        lay_wise_weights = cal_contribution_of_clients(global_model, temp_locals, val_dls, device,  layer_names)

        communication_ours(global_model, temp_locals, lay_wise_weights)

        global_w = global_model.state_dict()
        for idx, client in enumerate(clients):
            local_models[idx].load_state_dict(global_w)


        if r% args.test_step == 0:
            #test
            avg_loss = []
            avg_dice = []
            for idx, client in enumerate(clients):
                loss, dice = test(local_models[idx], test_dls[idx], device, loss_fun)

                logging.info('client: %s  test_loss:  %f   test_acc:  %f '%(client, loss, dice))
                avg_dice.append(dice)
                avg_loss.append(loss)

            avg_dice_v = sum(avg_dice) / len(avg_dice)
            avg_loss_v = sum(avg_loss) / len(avg_loss)

            logging.info('Round: [%d]  avg_test_loss: %f avg_test_acc: %f std_test_acc: %f'%(r, avg_loss_v, avg_dice_v, np.std(np.array(avg_dice))))

            if best_dice < avg_dice_v:
                best_dice = avg_dice_v
                best_dice_round = r
                best_local_dice = avg_dice

                weight_save_path = os.path.join(weight_save_dir, 'best.pth')
                torch.save(global_model.state_dict(), weight_save_path)
            

    logging.info('-------- Training complete --------')
    logging.info('Best avg dice score %f at round %d '%( best_dice, best_dice_round))
    for idx, client in enumerate(clients):
        logging.info('client: %s  test_acc:  %f '%(client, best_local_dice[idx]))


# if __name__ == '__main__':
#     args = get_args()
#     main(args)






In [21]:
args = get_args()

In [22]:
main(args)

2025-11-15 13:32:59 ===> Namespace(seed=0, data_root='/kaggle/input/fundus/fundus', dataset='fundus', model='unet', rounds=200, epochs=1, device='cuda', log_dir='./logs/', save_dir='./weights/', lr=0.001, weight_decay=0.0001, batch_size=8, experiment='experiment', test_step=1, train_ratio=0.6, cka_type='linear')
2025-11-15 13:32:59 ===> Namespace(seed=0, data_root='/kaggle/input/fundus/fundus', dataset='fundus', model='unet', rounds=200, epochs=1, device='cuda', log_dir='./logs/', save_dir='./weights/', lr=0.001, weight_decay=0.0001, batch_size=8, experiment='experiment', test_step=1, train_ratio=0.6, cka_type='linear')
2025-11-15 13:32:59 ===> Domain1 train  dataset: 40
2025-11-15 13:32:59 ===> Domain1 train  dataset: 40
2025-11-15 13:32:59 ===> Domain1 val  dataset: 10
2025-11-15 13:32:59 ===> Domain1 val  dataset: 10
2025-11-15 13:32:59 ===> Domain1 test  dataset: 51
2025-11-15 13:32:59 ===> Domain1 test  dataset: 51
2025-11-15 13:32:59 ===> Domain2 train  dataset: 79
2025-11-15 13:

total 40 slices
total 10 slices
total 51 slices
total 79 slices
total 20 slices
total 60 slices
total 256 slices
total 64 slices
total 80 slices
total 256 slices
total 64 slices
total 80 slices


2025-11-15 13:33:00 ===> Client: [0]  Epoch: [0]  train_loss: 0.508272 train_acc: 0.602970
2025-11-15 13:33:00 ===> Client: [0]  Epoch: [0]  train_loss: 0.508272 train_acc: 0.602970
2025-11-15 13:33:04 ===> Client: [1]  Epoch: [0]  train_loss: 0.495887 train_acc: 0.620059
2025-11-15 13:33:04 ===> Client: [1]  Epoch: [0]  train_loss: 0.495887 train_acc: 0.620059
2025-11-15 13:33:15 ===> Client: [2]  Epoch: [0]  train_loss: 0.356291 train_acc: 0.812351
2025-11-15 13:33:15 ===> Client: [2]  Epoch: [0]  train_loss: 0.356291 train_acc: 0.812351
2025-11-15 13:33:27 ===> Client: [3]  Epoch: [0]  train_loss: 0.416125 train_acc: 0.797989
2025-11-15 13:33:27 ===> Client: [3]  Epoch: [0]  train_loss: 0.416125 train_acc: 0.797989


41 Hooks registered. Total hooks: 41
41 Hooks registered. Total hooks: 41


Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 2:   0%|          | 0/8 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/8 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/8 [00:00<?, ?it/s]

41 handles removed.
41 handles removed.


2025-11-15 17:45:10 ===> client: Domain1  test_loss:  0.177917   test_acc:  0.808103 
2025-11-15 17:45:10 ===> client: Domain1  test_loss:  0.177917   test_acc:  0.808103 
2025-11-15 17:45:11 ===> client: Domain2  test_loss:  0.232406   test_acc:  0.721995 
2025-11-15 17:45:11 ===> client: Domain2  test_loss:  0.232406   test_acc:  0.721995 
2025-11-15 17:45:13 ===> client: Domain3  test_loss:  0.092713   test_acc:  0.903268 
2025-11-15 17:45:13 ===> client: Domain3  test_loss:  0.092713   test_acc:  0.903268 
2025-11-15 17:45:14 ===> client: Domain4  test_loss:  0.098739   test_acc:  0.901183 
2025-11-15 17:45:14 ===> client: Domain4  test_loss:  0.098739   test_acc:  0.901183 
2025-11-15 17:45:14 ===> Round: [199]  avg_test_loss: 0.150444 avg_test_acc: 0.833637 std_test_acc: 0.075045
2025-11-15 17:45:14 ===> Round: [199]  avg_test_loss: 0.150444 avg_test_acc: 0.833637 std_test_acc: 0.075045
2025-11-15 17:45:14 ===> -------- Training complete --------
2025-11-15 17:45:14 ===> --------

In [18]:
import os

root = "/kaggle/input/fundus/"
for path, dirs, files in os.walk(root):
    print(path, len(files))


/kaggle/input/fundus/ 0
/kaggle/input/fundus/fundus 0
/kaggle/input/fundus/fundus/Domain2 0
/kaggle/input/fundus/fundus/Domain2/test 0
/kaggle/input/fundus/fundus/Domain2/test/mask 60
/kaggle/input/fundus/fundus/Domain2/test/image 60
/kaggle/input/fundus/fundus/Domain2/train 0
/kaggle/input/fundus/fundus/Domain2/train/mask 99
/kaggle/input/fundus/fundus/Domain2/train/image 99
/kaggle/input/fundus/fundus/Domain3 0
/kaggle/input/fundus/fundus/Domain3/test 0
/kaggle/input/fundus/fundus/Domain3/test/mask 80
/kaggle/input/fundus/fundus/Domain3/test/image 80
/kaggle/input/fundus/fundus/Domain3/train 0
/kaggle/input/fundus/fundus/Domain3/train/mask 320
/kaggle/input/fundus/fundus/Domain3/train/image 320
/kaggle/input/fundus/fundus/Domain4 0
/kaggle/input/fundus/fundus/Domain4/test 0
/kaggle/input/fundus/fundus/Domain4/test/mask 80
/kaggle/input/fundus/fundus/Domain4/test/image 80
/kaggle/input/fundus/fundus/Domain4/train 0
/kaggle/input/fundus/fundus/Domain4/train/mask 320
/kaggle/input/fundu